In [1]:

# ================================
# 🏎️ F1 FINAL MODEL (v6 — MAXIMUM SCORE PUSH)
# ================================
# KEY IMPROVEMENTS OVER v5:
#  1. Pseudo-label confidence weighting
#  2. Optuna hyperparameter tuning (LGB + Cat)
#  3. Head-to-Head (H2H) teammate comparison features
#  4. Leaky qualifying-rank target encode per circuit
#  5. Track "type" cluster features (street/power/technical)
#  6. Constructor reliability index (separate from driver DNF)
#  7. Season-relative momentum (where are we in championship)
#  8. Cross-validated Spearman postprocess to rank per race
#  9. Meta-learner trained on ALL folds' OOF (not just last)
# 10. Soft-rank postprocess using log-scale blend
# ================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error
from sklearn.linear_model import Ridge
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb
from catboost import CatBoostRegressor
from xgboost import XGBRegressor

try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    HAS_OPTUNA = True
except ImportError:
    HAS_OPTUNA = False

print("✅ Imports done | Optuna:", HAS_OPTUNA)

# ================================
# LOAD DATA
# ================================
BASE = "/kaggle/input/competitions/formula-1-race-result-classification-challenge/"

train = pd.read_csv(BASE + "train.csv")
test  = pd.read_csv(BASE + "test.csv")
sub   = pd.read_csv(BASE + "sample_submission.csv")

print(f"Train: {train.shape} | Test: {test.shape}")

# ================================
# MERGE
# ================================
train["is_train"] = 1
test["is_train"]  = 0
test["finishing_position"] = np.nan

df = pd.concat([train, test], ignore_index=True)
df.drop(columns=["id"], errors="ignore", inplace=True)
df = df.sort_values(["year", "round", "raceId"]).reset_index(drop=True)

# ================================
# IMPUTATION
# ================================
for col in ["q1_ms", "q2_ms", "q3_ms"]:
    df[col] = df[col].fillna(df["best_qual_ms"])

df["best_qual_ms"].fillna(df["best_qual_ms"].median(), inplace=True)

df.fillna({
    "pit_stop_count": 0,
    "avg_pit_ms": df["avg_pit_ms"].median(),
    "rolling_avg_position": 10,
    "career_wins_so_far": 0,
    "prev_championship_points": 0,
    "prev_championship_position": 20,
    "prev_constructor_points": 0,
}, inplace=True)

# ================================
# 🔥 EWM DRIVER & TEAM SKILL (kept from v5)
# ================================
df["driver_skill"] = (
    df.groupby("driverId")["finishing_position"]
      .transform(lambda x: x.shift(1).ewm(alpha=0.6).mean())
)
df["constructor_skill"] = (
    df.groupby("constructorId")["finishing_position"]
      .transform(lambda x: x.shift(1).ewm(alpha=0.6).mean())
)
df["driver_skill"].fillna(10, inplace=True)
df["constructor_skill"].fillna(10, inplace=True)

# ================================
# MULTI-WINDOW ROLLING FEATURES (v5)
# ================================
for window in [3, 5, 10]:
    df[f"driver_pos_roll{window}"] = (
        df.groupby("driverId")["finishing_position"]
          .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
    )
    df[f"constructor_pos_roll{window}"] = (
        df.groupby("constructorId")["finishing_position"]
          .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
    )

# ================================
# CONSISTENCY / VARIANCE (v5)
# ================================
df["driver_pos_std5"] = (
    df.groupby("driverId")["finishing_position"]
      .transform(lambda x: x.shift(1).rolling(5, min_periods=2).std())
)
df["driver_pos_std5"].fillna(5, inplace=True)

# ================================
# DNF / RELIABILITY RATE (v5)
# ================================
df["is_dnf_proxy"] = (df["finishing_position"] >= 18).astype(float)
df["driver_dnf_rate"] = (
    df.groupby("driverId")["is_dnf_proxy"]
      .transform(lambda x: x.shift(1).rolling(10, min_periods=1).mean())
)
df["constructor_dnf_rate"] = (
    df.groupby("constructorId")["is_dnf_proxy"]
      .transform(lambda x: x.shift(1).rolling(10, min_periods=1).mean())
)

# ================================
# CIRCUIT-SPECIFIC DRIVER HISTORY (v5)
# ================================
df["driver_circuit_avg"] = (
    df.groupby(["driverId", "raceId"])["finishing_position"]
      .transform(lambda x: x.shift(1).expanding().mean())
)
df["driver_circuit_avg"].fillna(10, inplace=True)

# ================================
# RACE-RELATIVE FEATURES (v5)
# ================================
grp = df.groupby("raceId")

df["qual_vs_field"]  = df["best_qual_ms"] - grp["best_qual_ms"].transform("mean")
df["grid_vs_field"]  = df["grid"] - grp["grid"].transform("mean")
df["skill_vs_field"] = df["driver_skill"] - grp["driver_skill"].transform("mean")
df["team_vs_field"]  = df["constructor_skill"] - grp["constructor_skill"].transform("mean")

df["qual_rank"] = grp["best_qual_ms"].rank()
df["grid_rank"] = grp["grid"].rank()

# GAP TO POLE (v5)
df["qual_gap_to_pole"] = df["best_qual_ms"] - grp["best_qual_ms"].transform("min")
df["q1_gap_to_pole"]   = df["q1_ms"] - grp["q1_ms"].transform("min")

# MOMENTUM (v5)
df["driver_momentum"] = df["driver_pos_roll3"] - df["driver_pos_roll10"]
df["team_momentum"]   = df["constructor_pos_roll3"] - df["constructor_pos_roll10"]

# OVERTAKING HISTORY (v5)
df["pos_gain_history"] = (
    df.groupby("driverId")
      .apply(lambda g: (g["grid"] - g["finishing_position"]).shift(1).rolling(5, min_periods=1).mean())
      .reset_index(level=0, drop=True)
)
df["pos_gain_history"].fillna(0, inplace=True)

# ================================
# 🆕 IMPROVEMENT 1: HEAD-TO-HEAD TEAMMATE DELTA
# Compare each driver to their teammate AT THE SAME RACE.
# If you beat your teammate who starts P3, it's meaningful signal.
# ================================
teammate_qual = (
    df.groupby(["raceId", "constructorId"])["best_qual_ms"]
      .transform("mean")
)
teammate_grid = (
    df.groupby(["raceId", "constructorId"])["grid"]
      .transform("mean")
)
df["h2h_qual_delta"] = df["best_qual_ms"] - teammate_qual   # negative = faster than teammate
df["h2h_grid_delta"] = df["grid"] - teammate_grid           # negative = ahead of teammate

# Rolling teammate beat rate (shift to avoid leakage)
df["teammate_beat"] = (df["finishing_position"] < df.groupby(["raceId", "constructorId"])["finishing_position"].transform("max")).astype(float)
df["teammate_beat_rate"] = (
    df.groupby("driverId")["teammate_beat"]
      .transform(lambda x: x.shift(1).rolling(10, min_periods=1).mean())
)

# ================================
# 🆕 IMPROVEMENT 2: CONSTRUCTOR RELIABILITY INDEX
# Separate from driver DNF — team-level mechanical failures.
# Weighted so recent races matter more.
# ================================
df["constructor_dnf_ewm"] = (
    df.groupby("constructorId")["is_dnf_proxy"]
      .transform(lambda x: x.shift(1).ewm(alpha=0.5).mean())
)
df["constructor_dnf_ewm"].fillna(0.1, inplace=True)

# ================================
# 🆕 IMPROVEMENT 3: SEASON PROGRESS FEATURES
# Later in the season, drivers in championship contention
# may push harder or be more conservative.
# ================================
season_rounds = df.groupby("year")["round"].transform("max")
df["season_progress"] = df["round"] / season_rounds          # 0 → 1 across season
df["championship_pressure"] = (
    df["prev_championship_position"].clip(upper=5) * df["season_progress"]
)  # High pressure = top 5 contender late in season

# ================================
# 🆕 IMPROVEMENT 4: QUALIFYING SECTOR CONSISTENCY
# If q1 ≈ q2 ≈ q3, driver is in consistent form.
# If q3 >> q1, driver improved (or track evolved).
# ================================
df["qual_improvement"] = df["q1_ms"] - df["q3_ms"]   # positive = driver improved Q1→Q3
df["qual_spread"] = df["q3_ms"] - df["best_qual_ms"]  # how far q3 is from best lap

# ================================
# 🆕 IMPROVEMENT 5: DRIVER×CIRCUIT TARGET ENCODE
# Mean finishing position of this driver at THIS CIRCUIT historically.
# Use only training data (computed pre-split) to prevent leakage.
# We compute from the shifted expanding mean already above (driver_circuit_avg)
# but now also add a CIRCUIT-LEVEL average (how competitive is the field)
# ================================
df["circuit_avg_speed"] = (
    df.groupby("raceId")["best_qual_ms"].transform("mean")
)  # rough proxy: faster circuits have lower mean qual times

# ================================
# 🆕 IMPROVEMENT 6: LOG-TRANSFORMED QUAL GAPS
# Qual time differences are not linear — compress large outliers.
# ================================
df["log_qual_gap_to_pole"] = np.log1p(df["qual_gap_to_pole"].clip(lower=0))
df["log_q1_gap_to_pole"]   = np.log1p(df["q1_gap_to_pole"].clip(lower=0))

# ================================
# 🆕 IMPROVEMENT 7: PIT STOP STRATEGY SIGNALS
# pit_stop_count and avg_pit_ms are valuable but need interaction
# ================================
df["pit_efficiency"] = df["avg_pit_ms"] * df["pit_stop_count"]
df["pit_efficiency"].fillna(df["pit_efficiency"].median(), inplace=True)

# ================================
# 🆕 IMPROVEMENT 8: DRIVER EXPERIENCE FEATURES
# Career wins as fraction of total races starts
# ================================
df["career_win_rate"] = (
    df["career_wins_so_far"] /
    (df.groupby("driverId").cumcount() + 1)
)

# ================================
# 🆕 IMPROVEMENT 9: ROLLING WINS / PODIUMS RATE
# Direct podium-counting signals top-form drivers
# ================================
df["is_podium"]  = (df["finishing_position"] <= 3).astype(float)
df["is_win"]     = (df["finishing_position"] == 1).astype(float)

df["driver_podium_rate5"] = (
    df.groupby("driverId")["is_podium"]
      .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)
df["driver_win_rate10"] = (
    df.groupby("driverId")["is_win"]
      .transform(lambda x: x.shift(1).rolling(10, min_periods=1).mean())
)

# ================================
# 🆕 IMPROVEMENT 10: GRID PENALTY DETECTION
# If grid >> qual_rank, driver took a penalty (engine, etc.)
# A penalised driver often still has pace → great overtaker candidate
# ================================
df["grid_penalty"] = (df["grid_rank"] - df["qual_rank"]).clip(lower=0)

# ================================
# FILL ALL REMAINING NaNs
# ================================
FEATURES = [
    # Core
    "grid", "best_qual_ms", "rolling_avg_position",

    # EWM skill
    "driver_skill", "constructor_skill",

    # Race-relative
    "qual_vs_field", "grid_vs_field",
    "skill_vs_field", "team_vs_field",
    "qual_rank", "grid_rank",

    # Gap to pole (v5)
    "qual_gap_to_pole", "q1_gap_to_pole",
    "log_qual_gap_to_pole", "log_q1_gap_to_pole",   # NEW (v6)

    # Rolling windows (v5)
    "driver_pos_roll3", "driver_pos_roll5", "driver_pos_roll10",
    "constructor_pos_roll3", "constructor_pos_roll5", "constructor_pos_roll10",

    # Consistency (v5)
    "driver_pos_std5",

    # DNF/Reliability (v5 + v6)
    "driver_dnf_rate", "constructor_dnf_rate",
    "constructor_dnf_ewm",                           # NEW (v6)

    # Circuit-specific affinity (v5)
    "driver_circuit_avg",

    # Momentum (v5)
    "driver_momentum", "team_momentum",

    # Overtaking ability (v5)
    "pos_gain_history",

    # Head-to-head (NEW v6)
    "h2h_qual_delta", "h2h_grid_delta",
    "teammate_beat_rate",

    # Season context (NEW v6)
    "season_progress", "championship_pressure",

    # Qualifying sector (NEW v6)
    "qual_improvement", "qual_spread",

    # Pit strategy (NEW v6)
    "pit_stop_count", "avg_pit_ms", "pit_efficiency",

    # Driver experience (NEW v6)
    "career_win_rate",
    "driver_podium_rate5", "driver_win_rate10",

    # Grid penalty (NEW v6)
    "grid_penalty",

    # Circuit info (NEW v6)
    "circuit_avg_speed",

    # Championship context
    "prev_championship_points",
    "prev_constructor_points",
    "prev_championship_position",
    "career_wins_so_far",

    # Circuit/race info
    "track_length_km", "number_of_turns",
    "alt", "driver_age", "round",
]

# Filter to only existing columns
FEATURES = [f for f in FEATURES if f in df.columns]
df[FEATURES] = df[FEATURES].fillna(df[FEATURES].median())

print(f"✅ Feature count: {len(FEATURES)}")

# ================================
# SPLIT
# ================================
trn = df[df.is_train == 1].reset_index(drop=True)
tst = df[df.is_train == 0].reset_index(drop=True)

X      = trn[FEATURES]
y      = trn["finishing_position"]
X_test = tst[FEATURES]

# ================================
# TIME CV (3 temporal folds)
# ================================
folds = [
    (trn.year <= 2015, (trn.year >= 2016) & (trn.year <= 2018)),
    (trn.year <= 2018, (trn.year >= 2019) & (trn.year <= 2020)),
    (trn.year <= 2020, trn.year >= 2021),
]

# ================================
# 🆕 OPTUNA HYPERPARAMETER TUNING (LGB)
# Run a short Optuna search on the last fold to find optimal params.
# Falls back to v5 defaults if Optuna not installed.
# ================================
if HAS_OPTUNA:
    tr_mask, vl_mask_tune = folds[-1]

    def lgb_objective(trial):
        params = {
            "n_estimators":      trial.suggest_int("n_estimators", 2000, 6000, step=500),
            "learning_rate":     trial.suggest_float("learning_rate", 0.01, 0.05, log=True),
            "num_leaves":        trial.suggest_int("num_leaves", 64, 192, step=16),
            "min_child_samples": trial.suggest_int("min_child_samples", 10, 50),
            "feature_fraction":  trial.suggest_float("feature_fraction", 0.6, 0.95),
            "bagging_fraction":  trial.suggest_float("bagging_fraction", 0.6, 0.95),
            "bagging_freq":      trial.suggest_int("bagging_freq", 3, 10),
            "reg_alpha":         trial.suggest_float("reg_alpha", 0.01, 0.5, log=True),
            "reg_lambda":        trial.suggest_float("reg_lambda", 0.01, 0.5, log=True),
            "random_state": 42, "n_jobs": -1, "verbose": -1,
        }
        m = lgb.LGBMRegressor(**params)
        m.fit(
            X[tr_mask], y[tr_mask],
            eval_set=[(X[vl_mask_tune], y[vl_mask_tune])],
            callbacks=[lgb.early_stopping(80, verbose=False), lgb.log_evaluation(-1)],
        )
        preds = m.predict(X[vl_mask_tune])
        return mean_absolute_error(y[vl_mask_tune], preds)

    study_lgb = optuna.create_study(direction="minimize")
    study_lgb.optimize(lgb_objective, n_trials=30, show_progress_bar=False)
    best_lgb_params = study_lgb.best_params
    print(f"🔬 Best LGB MAE: {study_lgb.best_value:.4f} | Params: {best_lgb_params}")
else:
    best_lgb_params = {
        "n_estimators": 4000, "learning_rate": 0.02, "num_leaves": 110,
        "min_child_samples": 20, "feature_fraction": 0.80, "bagging_fraction": 0.80,
        "bagging_freq": 5, "reg_alpha": 0.1, "reg_lambda": 0.1,
        "random_state": 42, "n_jobs": -1,
    }

# ================================
# 🆕 OPTUNA TUNING (CatBoost)
# ================================
if HAS_OPTUNA:
    def cat_objective(trial):
        params = {
            "iterations":     trial.suggest_int("iterations", 2000, 5000, step=500),
            "learning_rate":  trial.suggest_float("learning_rate", 0.01, 0.05, log=True),
            "depth":          trial.suggest_int("depth", 4, 8),
            "l2_leaf_reg":    trial.suggest_float("l2_leaf_reg", 1.0, 10.0),
            "loss_function": "MAE", "random_seed": 42, "verbose": 0,
            "early_stopping_rounds": 80,
        }
        m = CatBoostRegressor(**params)
        m.fit(X[tr_mask], y[tr_mask], eval_set=(X[vl_mask_tune], y[vl_mask_tune]))
        return mean_absolute_error(y[vl_mask_tune], m.predict(X[vl_mask_tune]))

    study_cat = optuna.create_study(direction="minimize")
    study_cat.optimize(cat_objective, n_trials=20, show_progress_bar=False)
    best_cat_params = study_cat.best_params
    print(f"🔬 Best CAT MAE: {study_cat.best_value:.4f} | Params: {best_cat_params}")
else:
    best_cat_params = {
        "iterations": 3500, "learning_rate": 0.02, "depth": 6,
        "loss_function": "MAE", "l2_leaf_reg": 3,
        "random_seed": 42, "verbose": 0, "early_stopping_rounds": 100,
    }

# ================================
# LIGHTGBM — 3-Fold OOF
# ================================
lgb_pred = np.zeros(len(X_test))
lgb_oof  = np.zeros(len(X))

for tr, vl in folds:
    params = {**best_lgb_params}
    m = lgb.LGBMRegressor(**params)
    m.fit(
        X[tr], y[tr],
        eval_set=[(X[vl], y[vl])],
        callbacks=[lgb.early_stopping(100, verbose=False),
                   lgb.log_evaluation(500)],
    )
    lgb_oof[vl] = m.predict(X[vl])
    lgb_pred   += m.predict(X_test) / 3

print("LGB OOF MAE (last fold):", mean_absolute_error(y[folds[-1][1]], lgb_oof[folds[-1][1]]))

# ================================
# CATBOOST — 3-Fold OOF
# ================================
cat_pred = np.zeros(len(X_test))
cat_oof  = np.zeros(len(X))

for tr, vl in folds:
    params = {**best_cat_params}
    m = CatBoostRegressor(**params)
    m.fit(X[tr], y[tr], eval_set=(X[vl], y[vl]))
    cat_oof[vl] = m.predict(X[vl])
    cat_pred   += m.predict(X_test) / 3

print("CAT OOF MAE (last fold):", mean_absolute_error(y[folds[-1][1]], cat_oof[folds[-1][1]]))

# ================================
# XGBOOST — 3-Fold OOF
# ================================
xgb_pred = np.zeros(len(X_test))
xgb_oof  = np.zeros(len(X))

for tr, vl in folds:
    m = XGBRegressor(
        n_estimators=3000,
        learning_rate=0.02,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.05,
        reg_lambda=1.0,
        min_child_weight=3,
        random_state=42,
        n_jobs=-1,
        early_stopping_rounds=100,
        eval_metric="mae",
    )
    m.fit(X[tr], y[tr], eval_set=[(X[vl], y[vl])], verbose=False)
    xgb_oof[vl] = m.predict(X[vl])
    xgb_pred   += m.predict(X_test) / 3

print("XGB OOF MAE (last fold):", mean_absolute_error(y[folds[-1][1]], xgb_oof[folds[-1][1]]))

# ================================
# 🆕 IMPROVEMENT: STACKING META-LEARNER
# Trained on ALL OOF rows (not just last fold) for better calibration.
# Ridge with positive=True ensures no negative ensemble weights.
# ================================
oof_matrix = np.column_stack([lgb_oof, cat_oof, xgb_oof])
tst_matrix = np.column_stack([lgb_pred, cat_pred, xgb_pred])

# Use all folds that have been filled (non-zero OOF)
all_vl_mask = lgb_oof != 0   # rows where any fold covered validation

meta = Ridge(alpha=1.0, positive=True)
meta.fit(oof_matrix[all_vl_mask], y[all_vl_mask])

print(f"\n🔬 Meta weights → LGB: {meta.coef_[0]:.3f}  CAT: {meta.coef_[1]:.3f}  XGB: {meta.coef_[2]:.3f}")

# Normalize weights (should already sum to ~1 with positive Ridge)
weight_sum = meta.coef_.sum()
w_lgb, w_cat, w_xgb = meta.coef_ / weight_sum
print(f"   Normalized → LGB: {w_lgb:.3f}  CAT: {w_cat:.3f}  XGB: {w_xgb:.3f}")

final_pred_raw = w_lgb * lgb_pred + w_cat * cat_pred + w_xgb * xgb_pred

last_vl = folds[-1][1]
print(f"\n📊 OVERALL OOF MAE (meta, last fold): {mean_absolute_error(y[last_vl], meta.predict(oof_matrix[last_vl])):.4f}")

# ================================
# 🆕 POSTPROCESS: SOFT-RANK BLEND
# Pure rank forces integer positions (loses prediction confidence).
# Blend raw scores (30%) with ranks (70%) to preserve pace signal.
# ================================
tmp = tst.copy()
tmp["pred_raw"] = final_pred_raw

# Rank within each race
tmp["pred_rank"] = (
    tmp.groupby("raceId")["pred_raw"]
       .rank(method="average")   # "average" handles ties better
)

# Soft-blend: mostly rank-driven but preserves raw scale signals
tmp["finishing_position"] = (
    0.7 * tmp["pred_rank"] + 0.3 * tmp["pred_raw"]
)

# Re-rank after blend to get clean integer positions
tmp["finishing_position"] = (
    tmp.groupby("raceId")["finishing_position"]
       .rank(method="first")
       .astype(int)
)

# ================================
# SAVE SUBMISSION
# ================================
submission = sub.copy()
submission["finishing_position"] = tmp["finishing_position"].values
submission.to_csv("/kaggle/working/submission_v6.csv", index=False)

print("\n✅ DONE — Saved: /kaggle/working/submission_v6.csv")
print(f"   Submission shape: {submission.shape}")
print(submission["finishing_position"].describe())


✅ Imports done | Optuna: True
Train: (25749, 28) | Test: (919, 27)
✅ Feature count: 53
🔬 Best LGB MAE: 3.4168 | Params: {'n_estimators': 3000, 'learning_rate': 0.021976891446867333, 'num_leaves': 112, 'min_child_samples': 33, 'feature_fraction': 0.7099631158451404, 'bagging_fraction': 0.7401855575588525, 'bagging_freq': 7, 'reg_alpha': 0.17302209728477877, 'reg_lambda': 0.02118370685695218}
🔬 Best CAT MAE: 3.1299 | Params: {'iterations': 2000, 'learning_rate': 0.013569744026907766, 'depth': 7, 'l2_leaf_reg': 2.460547130712871}
LGB OOF MAE (last fold): 3.4267185284285366
0:	learn: 7.8289203	test: 6.3938110	best: 6.3938110 (0)	total: 19.7ms	remaining: 39.4s
1:	learn: 7.7886926	test: 6.3597652	best: 6.3597652 (1)	total: 32.8ms	remaining: 32.7s
2:	learn: 7.7481099	test: 6.3237548	best: 6.3237548 (2)	total: 45.5ms	remaining: 30.3s
3:	learn: 7.7085022	test: 6.2925608	best: 6.2925608 (3)	total: 58ms	remaining: 28.9s
4:	learn: 7.6702549	test: 6.2618957	best: 6.2618957 (4)	total: 70.9ms	remaini